<a href="https://colab.research.google.com/github/Feellived/molecular-reliability-signals/blob/yoonsoo/04_generate_variants(0917).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A-3. 변형 재생성 (2026-09-17)

`generate_variants.py` 를 A-3 개정본으로 바꿔 `variants_v2` 를 만든다.
담당4 실행 안내의 지시 그대로 세 곳을 고쳤다.

| # | 변경 | 근거 | 비용 |
|---|---|---|---|
| 1 | `A_AXIS_K` 10 → **30** | 최대 이탈 같은 꼬리 통계는 표본 10개면 부정확 | 채점 약 3배 |
| 2 | pH 창 6.4~8.4 → **5.0~9.0**, `max_variants` 8 → **12** | 위장관 pH 가 1~8 을 오감 | 변형 증가 |
| 3 | B3 **부분 입체 제거** | 분자당 변형 1개 → 중심 개수만큼 | **+2%** |

나머지는 원본 그대로다. A축 정준형 왕복 검사, `MAX_TAUTOMERS = 20`,
축 구성, 분할 상속, 표준화 재적용 금지 가드 전부 손대지 않았다.

In [ ]:
# ============================================================================
# [셀 1] 패키지 설치
# ----------------------------------------------------------------------------
# dimorphite-dl 은 양성자화 상태 열거에, rdkit 은 나머지 전부에 쓴다.
# ============================================================================
%pip install -q rdkit dimorphite-dl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.5 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# [셀 2] 드라이브 마운트 + 경로
# ----------------------------------------------------------------------------
# PROC_DIR : 분할 산출물. 이 아래에 <물성>/splits.csv 와 reports/ 가 있다.
# OUT_DIR  : 변형 출력. 기존 variants_role3 를 덮지 않도록 새 폴더로 낸다.
# WORK_DIR : 모듈을 쓸 로컬 폴더. 드라이브가 아니어야 임포트가 빠르다.
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, shutil
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROC_DIR = Path("/content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo")
OUT_DIR  = PROC_DIR / "variants_v2"
WORK_DIR = Path("/content/scripts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("분할 산출물:", PROC_DIR, "(있음)" if PROC_DIR.exists() else "(없음 - 경로 확인)")
print("변형 출력  :", OUT_DIR)
print("작업 폴더  :", WORK_DIR)

Mounted at /content/drive
분할 산출물: /content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo (있음)
변형 출력  : /content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo/variants_v2
작업 폴더  : /content/scripts


## 1단계 — 입력 점검

`generate_variants.py` 는 허용성 표를 **직접 읽지 않는다.**
`reports/07_axis_decision.csv` 의 `use_*` 열만 보고 `== "사용"` 으로 판정한다.
허용성 표는 그 위 단계(노트북 02)에서 07 을 만들 때 쓰인다.

In [ ]:
# ============================================================================
# [셀 3] 07_axis_decision.csv 와 splits.csv 점검
# ============================================================================
reports = PROC_DIR / "reports"
axis_csv = reports / "07_axis_decision.csv"

print("reports 폴더:", reports.exists())
for p in sorted(reports.glob("0[67]_*")) if reports.exists() else []:
    print("   ", p.name)
print()

assert axis_csv.exists(), f"없음: {axis_csv}  (노트북 02 로 먼저 생성해야 한다)"

dec = pd.read_csv(axis_csv)
print(f"07_axis_decision.csv : {len(dec)}행")
for c in ("use_B1_tautomer", "use_B1_protonation", "use_B2_salt", "use_B3_stereo"):
    if c in dec.columns:
        n = int((dec[c] == "사용").sum())
        print(f"  {c:22s} 사용 {n:2d} / {len(dec)}")
print()

have = [d.name for d in sorted(PROC_DIR.iterdir())
        if d.is_dir() and (d / "splits.csv").exists()]
print(f"splits.csv 보유: {len(have)}종")
missing = sorted(set(dec["dataset"]) - set(have))
if missing:
    print("  [주의] 07 에 있으나 splits.csv 없음:", missing)

reports 폴더: True
    06_allowance_rules.csv
    06_transformation_allowance_final.csv
    06_transformation_allowance_final.xlsx
    06_transformation_allowance_revised.csv
    07_axis_decision.csv

07_axis_decision.csv : 22행
  use_B1_tautomer        사용 22 / 22
  use_B1_protonation     사용 22 / 22
  use_B2_salt            사용  0 / 22
  use_B3_stereo          사용 19 / 22

splits.csv 보유: 22종


## 2단계 — 개정본 모듈을 파일로 쓴다

셀에 함수를 흩어 놓지 않고 **모듈 파일로 쓴 뒤 실행**한다. 이유가 둘이다.

1. `multiprocessing.Pool` 이 워커에서 함수를 임포트해야 하는데, 노트북 셀에
   정의된 함수는 그 경로가 불안정하다
2. 이 파일을 그대로 담당4 저장소의 `scripts/` 에 넣을 수 있어야 한다

`dataset_repairs.py` 는 담당4 저장소 파일이다. 없으면 무동작 대체본을 만든다 —
우리 `splits.csv` 는 파싱 실패 0행임을 확인했으므로
(`solubility_aqsoldb__9144`·`9145` 정정 완료) 이 함수가 할 일이 없다.

In [ ]:
%%writefile /content/scripts/generate_variants.py
#!/usr/bin/env python
"""A축·B축 변형 분자 생성 (연구계획서 5.3절).

입력은 담당1이 확정한 분할 파일이고, 출력은 물성별 변형 목록이다.
변형은 meta·test 분할에만 생성한다. train은 모델 적합에만 쓰이므로
신뢰성 신호를 산출할 필요가 없다.

축 구성

  A            표현 불안정성. 같은 분자를 서로 다른 SMILES 문자열로 쓴다.
               분자 상태는 고정하고 표기만 바꾼다.
  B1_tautomer  호변이성질체. 상태를 바꾸고 표기는 정규형으로 고정한다.
  B1_protonation  양성자화 상태. 마찬가지로 표기는 정규형으로 고정한다.
  B3_stereo    입체 표기. 입체 주석이 있는 분자에서 주석을 제거한 형태.

A축은 표기만, B축은 상태만 바꾼다. 이 분리가 두 축을 구분하는 근거이므로
B축 변형은 정규 SMILES로 기록하고 A축 변형은 정규화하지 않는다.

B2(염 형태)는 생성하지 않는다. 담당1의 07_axis_decision.csv에서 22종 모두
'표본부족' 또는 '허용안됨'으로 판정되어 사용 대상이 아니다.

지켜야 할 규율

  1. 변형 행은 원본 행의 split과 cv_fold를 그대로 상속한다.
     변형이 분할 경계를 넘으면 누출이 된다.
  2. 변형 생성 후 표준화(Cleanup, LargestFragmentChooser 등)를 재적용하지
     않는다. 우리가 만든 변이를 되돌려버린다.
  3. 각 변형은 parent_row_uid로 원본에 묶인다.

A-3 개정 (담당3). 원본 대비 바뀐 곳은 셋뿐이다.

  1. A_AXIS_K            10 → 30
  2. PH_MIN, PH_MAX      6.4, 8.4 → 5.0, 9.0   /  PROT_MAX_VARIANTS 8 → 12
  3. _gen_b3_stereo      전체 제거 → 입체중심별 부분 제거 + 전체 제거
"""

from __future__ import annotations

import argparse
import hashlib
import json
import multiprocessing as mp
import sys
import time
from pathlib import Path

import pandas as pd
import rdkit

sys.path.insert(0, str(Path(__file__).resolve().parent))
from dataset_repairs import apply_known_repairs  # noqa: E402
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

# 호변이성질체 상한은 eda_and_prescreen.py 및 게이트 판정과 같은 값을 쓴다.
# 이 값을 바꾸면 담당1의 B1 게이트 판정을 다시 계산해야 한다.
MAX_TAUTOMERS = 20

A_AXIS_K = 30                 # A-3: 10 → 30
PH_MIN = 5.0                  # A-3: 6.4 → 5.0
PH_MAX = 9.0                  # A-3: 8.4 → 9.0
PH_PRECISION = 1.0
PROT_MAX_VARIANTS = 12        # A-3: 8 → 12

TARGET_SPLITS = ("meta", "test")
AXES = ("A", "B1_tautomer", "B1_protonation", "B3_stereo")

OUT_COLUMNS = [
    "variant_uid",
    "parent_row_uid",
    "dataset",
    "task_type",
    "split",
    "cv_fold",
    "Y_final",
    "axis",
    "variant_index",
    "variant_smiles",
    "equals_parent",
]

_TAUTOMER_ENUMERATOR = None


def _init_worker() -> None:
    global _TAUTOMER_ENUMERATOR
    enumerator = rdMolStandardize.TautomerEnumerator()
    enumerator.SetMaxTautomers(MAX_TAUTOMERS)
    enumerator.SetRemoveSp3Stereo(False)
    enumerator.SetRemoveBondStereo(False)
    _TAUTOMER_ENUMERATOR = enumerator


def _seed_for(row_uid: str) -> int:
    """row_uid에서 결정적으로 난수 씨앗을 만든다. 재실행 시 같은 결과를 준다."""
    return int(hashlib.sha256(row_uid.encode("utf-8")).hexdigest()[:8], 16) % (2**31)


def _gen_a_axis(mol, seed: int, parent_canonical: str) -> tuple[list[str], int]:
    """같은 분자의 서로 다른 SMILES 표기. 정규화하지 않는다.

    무작위 표기는 원자 순서를 바꾸는데, 고리 안에 방향성 결합(/C=C\\)이 있는
    거대·고입체 분자에서는 이때 입체 표기가 어긋나는 경우가 드물게 생긴다.
    그런 변형은 원본과 다른 이성질체이므로 A축의 전제(같은 분자, 다른 표기)를
    깬다. 정규 SMILES로 되돌렸을 때 원본과 일치하는 것만 남긴다.

    되돌린 결과가 원본 정규형과 같다는 것이 곧 같은 분자라는 뜻이므로,
    이 검사는 올바른 변형을 잘못 버리지 않는다.

    A-3 주의: K를 30으로 올려도 이 검사는 유지한다. 빼면 A축 안에 B3(입체)
    효과가 섞여 두 축의 분리 근거가 무너진다.
    """
    drawn = Chem.MolToRandomSmilesVect(mol, A_AXIS_K * 3, randomSeed=seed)
    seen, out, n_rejected = set(), [], 0
    for smi in drawn:
        if smi in seen:
            continue
        roundtrip = Chem.MolFromSmiles(smi)
        if roundtrip is None or Chem.MolToSmiles(roundtrip) != parent_canonical:
            n_rejected += 1
            continue
        seen.add(smi)
        out.append(smi)
        if len(out) >= A_AXIS_K:
            break
    return out, n_rejected


def _gen_b1_tautomer(mol) -> list[str]:
    try:
        tautomers = _TAUTOMER_ENUMERATOR.Enumerate(mol)
    except Exception:
        return []
    seen, out = set(), []
    for taut in tautomers:
        try:
            smi = Chem.MolToSmiles(taut)
        except Exception:
            continue
        if smi and smi not in seen:
            seen.add(smi)
            out.append(smi)
    return out


def _gen_b1_protonation(smiles: str) -> list[str]:
    from dimorphite_dl import protonate_smiles

    try:
        raw = protonate_smiles(
            smiles,
            ph_min=PH_MIN,
            ph_max=PH_MAX,
            precision=PH_PRECISION,
            max_variants=PROT_MAX_VARIANTS,
        )
    except Exception:
        return []
    seen, out = set(), []
    for smi in raw:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        canon = Chem.MolToSmiles(mol)
        if canon not in seen:
            seen.add(canon)
            out.append(canon)
    return out


def _gen_b3_stereo(mol, parent_canonical: str) -> list[str]:
    """입체 주석 제거. 중심을 하나씩 지운 형태와 전부 지운 형태를 함께 만든다.

    개정 전에는 RemoveStereochemistry로 전부 지워 변형이 분자당 1개였다.
    흩어짐을 재는 축인데 표본이 점 하나여서, 원본을 넣어야 겨우 두 점이 되고
    그때의 표준편차는 원본과의 거리를 절반으로 나눈 값에 지나지 않았다.

    지정된 입체중심(includeUnassigned=False)을 하나씩만 지우면 중심 개수만큼
    변형이 나온다. 전부 지운 형태도 함께 남긴다.

    중복 제거가 필수다. 중심이 1개인 분자는 '하나 지우기'와 '전부 지우기'의
    결과가 같아서, 거르지 않으면 같은 SMILES가 두 번 들어가 표준편차가
    인위적으로 0에 눌린다.

    FindMolChiralCenters는 사면체 중심만 세므로 이중결합 기하(E/Z)만 가진
    분자는 전부 지운 형태 1개만 나온다. 부분 제거의 혜택은 사면체 중심을
    2개 이상 가진 분자(meta+test 19종 실측 61.7%)에 한정된다.
    """
    out: list[str] = []
    seen: set[str] = set()

    def _add(candidate) -> None:
        # 태그만 지우면 SMILES에 반영되지 않는 경우가 있어 다시 매긴다.
        try:
            Chem.AssignStereochemistry(candidate, cleanIt=True, force=True)
            smi = Chem.MolToSmiles(candidate)
        except Exception:
            return
        if smi and smi != parent_canonical and smi not in seen:
            seen.add(smi)
            out.append(smi)

    try:
        centers = Chem.FindMolChiralCenters(mol, includeUnassigned=False)
    except Exception:
        centers = []

    for idx, _cfg in centers:
        partial = Chem.Mol(mol)
        partial.GetAtomWithIdx(idx).SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
        _add(partial)

    flat = Chem.Mol(mol)
    Chem.RemoveStereochemistry(flat)
    _add(flat)

    return out


def _process_row(task: tuple) -> tuple[list[dict], dict]:
    (
        row_uid,
        dataset,
        task_type,
        split,
        cv_fold,
        y_final,
        parent_smiles,
        use_b1_tautomer,
        use_b1_protonation,
        use_b3_stereo,
    ) = task

    stats = {"n_failed_parse": 0, "n_invalid_variant": 0, "n_a_axis_rejected": 0}
    mol = Chem.MolFromSmiles(parent_smiles)
    if mol is None:
        stats["n_failed_parse"] = 1
        return [], stats

    parent_canonical = Chem.MolToSmiles(mol)
    seed = _seed_for(row_uid)

    a_variants, n_a_rejected = _gen_a_axis(mol, seed, parent_canonical)
    stats["n_a_axis_rejected"] = n_a_rejected
    by_axis: dict[str, list[str]] = {"A": a_variants}
    by_axis["B1_tautomer"] = _gen_b1_tautomer(mol) if use_b1_tautomer else []
    by_axis["B1_protonation"] = (
        _gen_b1_protonation(parent_smiles) if use_b1_protonation else []
    )
    by_axis["B3_stereo"] = (
        _gen_b3_stereo(mol, parent_canonical) if use_b3_stereo else []
    )

    rows = []
    for axis in AXES:
        for index, smiles in enumerate(by_axis[axis]):
            if Chem.MolFromSmiles(smiles) is None:
                stats["n_invalid_variant"] += 1
                continue
            rows.append(
                {
                    "variant_uid": f"{row_uid}__{axis}__{index:03d}",
                    "parent_row_uid": row_uid,
                    "dataset": dataset,
                    "task_type": task_type,
                    "split": split,
                    "cv_fold": cv_fold,
                    "Y_final": y_final,
                    "axis": axis,
                    "variant_index": index,
                    "variant_smiles": smiles,
                    "equals_parent": smiles == parent_canonical,
                }
            )
    return rows, stats


def _load_axis_decision(reports_dir: Path) -> dict[str, dict[str, bool]]:
    table = pd.read_csv(reports_dir / "07_axis_decision.csv")
    decision = {}
    for _, row in table.iterrows():
        decision[row["dataset"]] = {
            "B1_tautomer": row["use_B1_tautomer"] == "사용",
            "B1_protonation": row["use_B1_protonation"] == "사용",
            "B2_salt": row["use_B2_salt"] == "사용",
            "B3_stereo": row["use_B3_stereo"] == "사용",
        }
    return decision


def _process_dataset(
    dataset: str, splits_path: Path, flags: dict[str, bool], out_dir: Path, workers: int,
    target_splits: tuple[str, ...] = TARGET_SPLITS,
) -> dict:
    splits = pd.read_csv(splits_path)
    splits, _ = apply_known_repairs(splits)
    target = splits[splits["split"].isin(target_splits)].copy()

    tasks = [
        (
            row["row_uid"],
            dataset,
            row["task_type"],
            row["split"],
            row["cv_fold"],
            row["Y_final"],
            row["parent_smiles"],
            flags["B1_tautomer"],
            flags["B1_protonation"],
            flags["B3_stereo"],
        )
        for _, row in target.iterrows()
    ]

    started = time.time()
    all_rows: list[dict] = []
    n_failed_parse = 0
    n_invalid_variant = 0
    n_a_axis_rejected = 0

    with mp.Pool(workers, initializer=_init_worker) as pool:
        for rows, stats in pool.imap_unordered(_process_row, tasks, chunksize=16):
            all_rows.extend(rows)
            n_failed_parse += stats["n_failed_parse"]
            n_invalid_variant += stats["n_invalid_variant"]
            n_a_axis_rejected += stats["n_a_axis_rejected"]

    frame = pd.DataFrame(all_rows, columns=OUT_COLUMNS)
    frame = frame.sort_values(["parent_row_uid", "axis", "variant_index"])

    dataset_dir = out_dir / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(dataset_dir / "variants.csv", index=False)

    per_axis = {axis: int((frame["axis"] == axis).sum()) for axis in AXES}

    # B3 부분 제거가 실제로 표본을 늘렸는지 보는 지표.
    # 1.0 이면 대부분 입체중심이 1개여서 개정 효과가 없다는 뜻이다.
    n_b3_parents = int(
        frame.loc[frame["axis"] == "B3_stereo", "parent_row_uid"].nunique()
    )
    b3_per_molecule = (
        round(per_axis["B3_stereo"] / n_b3_parents, 2) if n_b3_parents else 0.0
    )

    summary = {
        "dataset": dataset,
        "n_parent_molecules": len(target),
        "n_variants_total": len(frame),
        "elapsed_sec": round(time.time() - started, 1),
        "n_failed_parse": n_failed_parse,
        "n_invalid_variant": n_invalid_variant,
        "n_a_axis_rejected": n_a_axis_rejected,
        "n_b3_parent_molecules": n_b3_parents,
        "b3_variants_per_molecule": b3_per_molecule,
        **{f"n_{axis}": per_axis[axis] for axis in AXES},
        **{f"use_{axis}": flags[axis] for axis in ("B1_tautomer", "B1_protonation", "B3_stereo")},
    }
    return summary


def main() -> int:
    parser = argparse.ArgumentParser(description="A축·B축 변형 분자 생성")
    parser.add_argument("--processed-dir", required=True, help="담당1 분할 산출물 최상위")
    parser.add_argument("--out-dir", required=True, help="변형 출력 최상위")
    parser.add_argument("--datasets", nargs="*", default=None, help="일부만 처리할 때")
    parser.add_argument("--workers", type=int, default=max(1, mp.cpu_count() - 2))
    parser.add_argument("--resume", action="store_true", help="이미 만든 물성은 건너뛴다")
    parser.add_argument(
        "--target-splits", nargs="+", default=list(TARGET_SPLITS),
        help="변형을 만들 분할. 기본은 meta와 test다. 결합 규칙 학습 표본을 늘리려면 "
             "train을 추가하되, 그 경우 폴드 외 예측을 별도로 마련해야 한다",
    )
    args = parser.parse_args()

    processed_dir = Path(args.processed_dir)
    out_dir = Path(args.out_dir)
    reports_dir = processed_dir / "reports"

    decision = _load_axis_decision(reports_dir)
    datasets = args.datasets or sorted(decision)

    out_dir.mkdir(parents=True, exist_ok=True)
    summaries = []

    for i, dataset in enumerate(datasets, 1):
        splits_path = processed_dir / dataset / "splits.csv"
        if not splits_path.exists():
            print(f"[{i}/{len(datasets)}] {dataset}: splits.csv 없음, 건너뜀", flush=True)
            continue
        if args.resume and (out_dir / dataset / "variants.csv").exists():
            print(f"[{i}/{len(datasets)}] {dataset}: 이미 있음, 건너뜀", flush=True)
            continue

        flags = decision[dataset]
        summary = _process_dataset(
            dataset, splits_path, flags, out_dir, args.workers, tuple(args.target_splits)
        )
        summaries.append(summary)
        print(
            f"[{i}/{len(datasets)}] {dataset}: "
            f"분자 {summary['n_parent_molecules']:,} → 변형 {summary['n_variants_total']:,} "
            f"(A {summary['n_A']:,} / 호변 {summary['n_B1_tautomer']:,} / "
            f"양성자 {summary['n_B1_protonation']:,} / "
            f"입체 {summary['n_B3_stereo']:,} [분자당 {summary['b3_variants_per_molecule']}]) "
            f"{summary['elapsed_sec']}초",
            flush=True,
        )

    if not summaries:
        print("새로 생성한 물성이 없다.")
        return 0

    summary_dir = out_dir / "_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_frame = pd.DataFrame(summaries)
    summary_frame.to_csv(summary_dir / "variant_summary.csv", index=False)

    from importlib.metadata import version as _pkg_version  # noqa: PLC0415

    try:
        dimorphite_version = _pkg_version("dimorphite_dl")
    except Exception:
        dimorphite_version = "unknown"

    metadata = {
        "generated_by": "yoonsoo (담당3, A-3 개정)",
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "source_splits": str(processed_dir),
        "target_splits": list(args.target_splits),
        "axes": list(AXES),
        "parameters": {
            "max_tautomers": MAX_TAUTOMERS,
            "a_axis_k": A_AXIS_K,
            "ph_min": PH_MIN,
            "ph_max": PH_MAX,
            "ph_precision": PH_PRECISION,
            "protonation_max_variants": PROT_MAX_VARIANTS,
            "b3_partial_stereo_removal": True,
        },
        "versions": {
            "rdkit": rdkit.__version__,
            "dimorphite_dl": dimorphite_version,
            "python": sys.version.split()[0],
        },
        "guards": [
            "변형 행은 원본의 split과 cv_fold를 상속한다",
            "변형 생성 후 표준화를 재적용하지 않는다",
            "B2(염 형태)는 게이트 판정에 따라 생성하지 않는다",
            "A축은 정규화하지 않고 B축은 정규 SMILES로 기록한다",
            "A축 변형은 정규 SMILES 왕복이 원본과 일치하는 것만 남긴다",
            "B3는 입체중심별 부분 제거와 전체 제거를 함께 내며 중복은 제거한다",
        ],
        "totals": {
            "n_datasets": len(summary_frame),
            "n_parent_molecules": int(summary_frame["n_parent_molecules"].sum()),
            "n_variants_total": int(summary_frame["n_variants_total"].sum()),
            "n_a_axis_rejected": int(summary_frame["n_a_axis_rejected"].sum()),
            **{
                f"n_{axis}": int(summary_frame[f"n_{axis}"].sum()) for axis in AXES
            },
        },
    }
    (summary_dir / "generation_metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    print()
    print(
        f"완료: {metadata['totals']['n_datasets']}종, "
        f"분자 {metadata['totals']['n_parent_molecules']:,} → "
        f"변형 {metadata['totals']['n_variants_total']:,}"
    )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/scripts/generate_variants.py


In [ ]:
# ============================================================================
# [셀 5] 모듈 작성 확인 + dataset_repairs.py 확보
# ============================================================================
assert WORK_DIR == Path("/content/scripts"), f"WORK_DIR({WORK_DIR}) 불일치"
module = WORK_DIR / "generate_variants.py"
assert module.exists(), "앞 셀(%%writefile)이 실행되지 않았다"
print("모듈 작성됨:", module, f"({len(module.read_text(encoding='utf-8').splitlines())}줄)")

# 우리 splits.csv 는 파싱 실패 0행이라 이 함수가 고칠 것이 없다.
# 드라이브 탐색은 없는 경로를 확인하는 데 오래 걸려 생략하고 바로 대체본을 쓴다.
(WORK_DIR / "dataset_repairs.py").write_text(
    '"""무동작 대체본. splits.csv 가 이미 정정된 상태라 고칠 것이 없다."""\n'
    "def apply_known_repairs(frame):\n"
    "    return frame, {}\n",
    encoding="utf-8",
)
print("dataset_repairs.py: 무동작 대체본")

for line in module.read_text(encoding="utf-8").splitlines():
    if line.startswith(("A_AXIS_K", "PH_MIN", "PH_MAX", "PROT_MAX_VARIANTS", "MAX_TAUTOMERS")):
        print("   ", line)

모듈 작성됨: /content/scripts/generate_variants.py (480줄)
dataset_repairs.py: 무동작 대체본
    MAX_TAUTOMERS = 20
    A_AXIS_K = 30                 # A-3: 10 → 30
    PH_MIN = 5.0                  # A-3: 6.4 → 5.0
    PH_MAX = 9.0                  # A-3: 8.4 → 9.0
    PROT_MAX_VARIANTS = 12        # A-3: 8 → 12


## 3단계 — 소규모 시험 실행

전체를 돌리기 전에 작은 물성 둘로 확인한다. 볼 것은 출력 줄의
**`입체 N [분자당 X]`** 하나다.

| X | 뜻 |
|---|---|
| **2 이상** | B3 부분 제거가 먹었다. 계속 진행 |
| **1.0 근처** | 대부분 입체중심이 1개다. B3 개정 효과 없음 |

`herg` 는 입체 분자 58개 중 33개가 중심 2개 이상이라 약 2.33배를 기대한다.
`vdss_lombardo` 는 입체 분자가 9개뿐이라 배수가 커도 신호로는 못 쓴다 —
동작 확인용이다.

In [ ]:
# ============================================================================
# [셀 6] 시험 실행 (2종)
# ============================================================================
cmd = (
    f'cd {WORK_DIR} && python generate_variants.py'
    f' --processed-dir "{PROC_DIR}"'
    f' --out-dir "{OUT_DIR}"'
    f' --datasets herg vdss_lombardo'
    f' --target-splits meta test'
    f' --workers 4'
)
print(cmd, "\n")
!{cmd}

cd /content/scripts && python generate_variants.py --processed-dir "/content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo" --out-dir "/content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo/variants_v2" --datasets herg vdss_lombardo --target-splits meta test --workers 4 

[1/2] herg: 분자 127 → 변형 5,183 (A 3,802 / 호변 577 / 양성자 634 / 입체 170 [분자당 2.79]) 6.0초
[2/2] vdss_lombardo: 분자 222 → 변형 9,519 (A 6,660 / 호변 1,457 / 양성자 1,291 / 입체 111 [분자당 4.27]) 10.9초

완료: 2종, 분자 349 → 변형 14,702


In [ ]:
# ============================================================================
# [셀 7] 시험 결과 확인 - 축별 분포와 B3 배수
# ============================================================================
for name in ("herg", "vdss_lombardo"):
    f = OUT_DIR / name / "variants.csv"
    if not f.exists():
        print(f"[없음] {name}")
        continue
    v = pd.read_csv(f)
    print(f"=== {name}  변형 {len(v):,}건")
    print(v["axis"].value_counts().to_string())

    b3 = v[v["axis"] == "B3_stereo"]
    if len(b3):
        per = b3.groupby("parent_row_uid").size()
        print(f"    B3 분자당 변형: 평균 {per.mean():.2f} | 중앙값 {per.median():.0f}"
              f" | 최대 {per.max()} | 분자 {len(per)}개")
        print(f"    분포: {per.value_counts().sort_index().to_dict()}")
        print("    ->", "부분 제거 작동" if per.mean() > 1.5 else "!! 효과 미미, 진행 재검토")
    print()

=== herg  변형 5,183건
axis
A                 3802
B1_protonation     634
B1_tautomer        577
B3_stereo          170
    B3 분자당 변형: 평균 2.79 | 중앙값 3 | 최대 16 | 분자 61개
    분포: {1: 26, 2: 3, 3: 15, 4: 7, 5: 7, 6: 1, 8: 1, 16: 1}
    -> 부분 제거 작동

=== vdss_lombardo  변형 9,519건
axis
A                 6660
B1_tautomer       1457
B1_protonation    1291
B3_stereo          111
    B3 분자당 변형: 평균 4.27 | 중앙값 1 | 최대 20 | 분자 26개
    분포: {1: 17, 3: 2, 4: 1, 8: 1, 12: 1, 13: 1, 14: 1, 17: 1, 20: 1}
    -> 부분 제거 작동



## 4단계 — 전체 실행

시험이 통과하면 22종 전체를 돌린다. `--resume` 이 붙어 있어 시험에서 만든
`herg`·`vdss_lombardo` 는 건너뛴다.

A축이 분자당 30개로 늘어 시간이 원본의 약 3배 걸린다.

In [ ]:
# ============================================================================
# [셀 8] 전체 실행 (22종)
# ============================================================================
cmd = (
    f'cd {WORK_DIR} && python generate_variants.py'
    f' --processed-dir "{PROC_DIR}"'
    f' --out-dir "{OUT_DIR}"'
    f' --target-splits meta test'
    f' --workers 8 --resume'
)
print(cmd, "\n")
!{cmd}

cd /content/scripts && python generate_variants.py --processed-dir "/content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo" --out-dir "/content/drive/MyDrive/MIST/data/processed/pipeline_yoonsoo/variants_v2" --target-splits meta test --workers 8 --resume 

[1/22] ames: 이미 있음, 건너뜀
[2/22] bbb_martins: 이미 있음, 건너뜀
[3/22] bioavailability_ma: 이미 있음, 건너뜀
[4/22] caco2_wang: 이미 있음, 건너뜀
[5/22] clearance_hepatocyte_az: 이미 있음, 건너뜀
[6/22] clearance_microsome_az: 이미 있음, 건너뜀
[7/22] cyp2c9_substrate_carbonmangels: 이미 있음, 건너뜀
[8/22] cyp2c9_veith: 이미 있음, 건너뜀
[9/22] cyp2d6_substrate_carbonmangels: 이미 있음, 건너뜀
[10/22] cyp2d6_veith: 이미 있음, 건너뜀
[11/22] cyp3a4_substrate_carbonmangels: 이미 있음, 건너뜀
[12/22] cyp3a4_veith: 이미 있음, 건너뜀
[13/22] dili: 이미 있음, 건너뜀
[14/22] half_life_obach: 이미 있음, 건너뜀
[15/22] herg: 이미 있음, 건너뜀
[16/22] hia_hou: 이미 있음, 건너뜀
[17/22] ld50_zhu: 이미 있음, 건너뜀
[18/22] lipophilicity_astrazeneca: 이미 있음, 건너뜀
[19/22] pgp_broccatelli: 이미 있음, 건너뜀
[20/22] ppbr_az: 이미 있음, 건너뜀
[21/22] solubility_aqsoldb: 

In [ ]:
# ============================================================================
# [셀 9] 전체 요약
# ============================================================================
summ = pd.read_csv(OUT_DIR / "_summary" / "variant_summary.csv")
meta = json.loads((OUT_DIR / "_summary" / "generation_metadata.json").read_text(encoding="utf-8"))

cols = [c for c in ("dataset", "n_parent_molecules", "n_A", "n_B1_tautomer",
                    "n_B1_protonation", "n_B3_stereo",
                    "b3_variants_per_molecule") if c in summ.columns]
print(summ[cols].to_string(index=False))
print()

t = meta["totals"]
print(f"총 변형 {t['n_variants_total']:,}건 / {t['n_datasets']}종")
for a in meta["axes"]:
    print(f"  {a:18s} {t['n_' + a]:>9,}")
print()
print(f"A축 왕복 검사 탈락: {t['n_a_axis_rejected']:,}")
print()
print("열거 파라미터:", json.dumps(meta["parameters"], ensure_ascii=False))

      dataset  n_parent_molecules  n_A  n_B1_tautomer  n_B1_protonation  n_B3_stereo  b3_variants_per_molecule
         herg                 127 3802            577               634          170                      2.79
vdss_lombardo                 222 6660           1457              1291          111                      4.27

총 변형 14,702건 / 2종
  A                     10,462
  B1_tautomer            2,034
  B1_protonation         1,925
  B3_stereo                281

A축 왕복 검사 탈락: 17

열거 파라미터: {"max_tautomers": 20, "a_axis_k": 30, "ph_min": 5.0, "ph_max": 9.0, "ph_precision": 1.0, "protonation_max_variants": 12, "b3_partial_stereo_removal": true}


In [ ]:
# ============================================================================
# [셀 10] 개정 전후 비교 - 기존 variants_role3 가 있을 때만
# ----------------------------------------------------------------------------
# 협업 규칙: 수치가 바뀌면 바뀐 값과 바뀌기 전 값을 함께 보고한다.
# ============================================================================
OLD_DIR = Path("/content/drive/MyDrive/MIST/Juhyeong/data/processed/variants_role3")

if not (OLD_DIR / "_summary" / "variant_summary.csv").exists():
    print("기존 산출물이 없어 비교를 건너뛴다:", OLD_DIR)
else:
    old = pd.read_csv(OLD_DIR / "_summary" / "variant_summary.csv")
    m = old.merge(summ, on="dataset", suffixes=("_old", "_new"))
    rows = []
    for axis in ("A", "B1_tautomer", "B1_protonation", "B3_stereo"):
        co, cn = f"n_{axis}_old", f"n_{axis}_new"
        if co in m.columns and cn in m.columns:
            a, b = int(m[co].sum()), int(m[cn].sum())
            rows.append({"축": axis, "개정 전": a, "개정 후": b,
                         "배수": round(b / a, 2) if a else None})
    print(pd.DataFrame(rows).to_string(index=False))

기존 산출물이 없어 비교를 건너뛴다: /content/drive/MyDrive/MIST/Juhyeong/data/processed/variants_role3


In [ ]:
print("개정 전 물성:", len(old), "| 개정 후 물성:", len(summ), "| 공통:", len(m))
print("개정 후에만 있음:", sorted(set(summ.dataset) - set(old.dataset)))
print("개정 전에만 있음:", sorted(set(old.dataset) - set(summ.dataset)))
print()
print("개정 후 분자 합계:", int(summ["n_parent_molecules"].sum()))

NameError: name 'old' is not defined

In [ ]:
import pandas as pd
from pathlib import Path

def axis_counts(root: Path) -> pd.DataFrame:
    """요약 파일을 믿지 않고 variants.csv 를 직접 세어 집계한다."""
    rows = []
    for p in sorted(root.iterdir()):
        f = p / "variants.csv"
        if not p.is_dir() or p.name == "_summary" or not f.exists():
            continue
        v = pd.read_csv(f, usecols=["axis", "parent_row_uid"])
        c = v["axis"].value_counts().to_dict()
        c["dataset"] = p.name
        c["n_parent"] = v["parent_row_uid"].nunique()
        b3 = v[v["axis"] == "B3_stereo"]
        c["b3_per_mol"] = round(len(b3) / b3["parent_row_uid"].nunique(), 2) if len(b3) else 0.0
        rows.append(c)
    return pd.DataFrame(rows).fillna(0)

new = axis_counts(OUT_DIR)
print(f"개정 후 물성 {len(new)}종")
cols = ["dataset", "n_parent", "A", "B1_tautomer", "B1_protonation", "B3_stereo", "b3_per_mol"]
print(new[[c for c in cols if c in new.columns]].to_string(index=False))
print()

missing = sorted(set(pd.read_csv(PROC_DIR / "reports" / "07_axis_decision.csv")["dataset"]) - set(new["dataset"]))
print("아직 안 만든 물성:", missing if missing else "없음 (22종 완료)")

In [ ]:
old = axis_counts(OLD_DIR)
m = old.merge(new, on="dataset", suffixes=("_old", "_new"))
print(f"비교 물성 {len(m)}종\n")

rows = []
for axis in ("A", "B1_tautomer", "B1_protonation", "B3_stereo"):
    co, cn = f"{axis}_old", f"{axis}_new"
    if co in m.columns and cn in m.columns:
        a, b = int(m[co].sum()), int(m[cn].sum())
        rows.append({"축": axis, "개정 전": a, "개정 후": b,
                     "배수": round(b / a, 2) if a else None})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

dist = {}
for p in sorted(OUT_DIR.iterdir()):
    f = p / "variants.csv"
    if not p.is_dir() or p.name == "_summary" or not f.exists():
        continue
    v = pd.read_csv(f, usecols=["axis", "parent_row_uid"])
    b3 = v[v["axis"] == "B3_stereo"]
    if len(b3):
        for k, n in b3.groupby("parent_row_uid").size().value_counts().items():
            dist[k] = dist.get(k, 0) + int(n)

tot = sum(dist.values())
one = dist.get(1, 0)
print(f"B3 보유 분자 {tot:,}개")
print(f"  변형 1개  {one:,} ({100*one/tot:.1f}%)   <- 개선 안 됨")
print(f"  변형 2개+ {tot-one:,} ({100*(tot-one)/tot:.1f}%)  <- 개선됨")
print(f"\n분포: {dict(sorted(dist.items()))}")

In [ ]:
# ============================================================================
# [셀 11] 산출물 저장 + MANIFEST
# ----------------------------------------------------------------------------
# 협업 규칙: 결과 데이터는 드라이브에 올리고 파일별 SHA-256 과 행 수를 적은
#            MANIFEST.csv 를 함께 넣는다.
# 원본 variant_summary.csv 는 덮지 않는다 (--resume 때문에 부분 집계지만
# 담당4 쪽 스크립트가 그 이름을 기대할 수 있다).
# ============================================================================
import hashlib
import pandas as pd
from pathlib import Path

SUM_DIR = OUT_DIR / "_summary"
SUM_DIR.mkdir(parents=True, exist_ok=True)

# 1) 물성별 전체 집계 (22종)
new.to_csv(SUM_DIR / "variant_summary_full.csv", index=False)

# 2) B3 분자당 변형 수 분포
b3_dist = (pd.DataFrame(sorted(dist.items()), columns=["variants_per_molecule", "n_molecules"])
           .assign(pct=lambda d: (100 * d.n_molecules / d.n_molecules.sum()).round(2)))
b3_dist.to_csv(SUM_DIR / "b3_variant_distribution.csv", index=False)

# 3) 개정 전후 비교
comp = []
for axis in ("A", "B1_tautomer", "B1_protonation", "B3_stereo"):
    co, cn = f"{axis}_old", f"{axis}_new"
    if co in m.columns and cn in m.columns:
        a, b = int(m[co].sum()), int(m[cn].sum())
        comp.append({"axis": axis, "before": a, "after": b,
                     "ratio": round(b / a, 2) if a else None})
comp_df = pd.DataFrame(comp)
comp_df.loc[len(comp_df)] = {"axis": "TOTAL",
                             "before": comp_df.before.sum(),
                             "after": comp_df.after.sum(),
                             "ratio": round(comp_df.after.sum() / comp_df.before.sum(), 2)}
comp_df.to_csv(SUM_DIR / "revision_comparison.csv", index=False)

print(comp_df.to_string(index=False))
print()

# 4) MANIFEST — variants.csv 전부 + 요약 파일
def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while (b := f.read(chunk)):
            h.update(b)
    return h.hexdigest()

rows = []
targets = sorted(OUT_DIR.glob("*/variants.csv")) + sorted(SUM_DIR.glob("*.csv"))
for p in targets:
    n = sum(1 for _ in open(p, encoding="utf-8")) - 1      # 헤더 제외
    rows.append({"path": str(p.relative_to(OUT_DIR)), "rows": n,
                 "bytes": p.stat().st_size, "sha256": sha256(p)})

man = pd.DataFrame(rows)
man.to_csv(OUT_DIR / "MANIFEST.csv", index=False)

print(f"MANIFEST: {len(man)}개 파일, 총 {man['rows'].sum():,}행, "
      f"{man['bytes'].sum()/1e6:.1f} MB")
print()
for p in sorted(SUM_DIR.glob("*.csv")):
    print("  ", p.name)
print("  ", (OUT_DIR / "MANIFEST.csv").name)

## 마치고 할 일

**1. 이어지는 단계** — 새 폴더로 2~7단계를 다시 돌리고 8단계로 판정한다.

```
2) 지문 채점    score_variants_fingerprint.py   --variants-dir variants_v2
3) 언어모델     score_variants_chemberta.py     --variants-dir variants_v2
4) 축 분산      compute_ab_variance.py
5) 신호 통합    assemble_signals.py
6) 조건부 B     build_conditional_signals.py
7) 확장 통계    build_rich_variant_features.py
8) 판정        run_preregistered_ablation.py
```

**2. 성공 기준** (담당4 지정)

- 축 기여가 AUPRC **+0.0225** 를 넘는가 → 주로 A축 30개가 답할 질문
- **B3 기여가 0에서 벗어나는가** → 현재 정규화 AURC 0.984 (거의 무작위)

** 전달 사항**

- 바뀐 곳 셋: `A_AXIS_K` 10→30 / pH 창 6.4~8.4→5.0~9.0 및 `max_variants` 8→12 /
  `_gen_b3_stereo` 부분 제거
- B3 는 60%만 고쳐진다 (중심 2개 이상 61.7%)

